# CheXFound Backbone Verification

**Model**: CheXFound — ViT-L/16, iBOT-style self-supervised pre-training  
**Arch**: ViT-Large/16 — 307M params  
**Embed dim**: 1024 (CLS token from ViT-L)  
**Input**: 3×512×512 px  
**Access**: Weights on Google Drive (request from GitHub); code on GitHub  
**Zero-shot**: No — vision-only

Paper: [CheXFound: Chest X-ray Foundation Model with Global and Local Representations Integration (arXiv 2502.05142)](https://arxiv.org/abs/2502.05142)

### Prerequisites

```bash
# 1. Clone the repo
git clone https://github.com/RPIDIAL/CheXFound /tmp/CheXFound
cd /tmp/CheXFound && pip install -r requirements.txt

# 2. Download pretrained weights from Google Drive:
#    https://drive.google.com/drive/folders/1GX2BWbujuVABtVpSZ4PTBykGULzrw806
#    File: teacher_checkpoint.pth
#    Place at: /tmp/CheXFound/weights/teacher_checkpoint.pth

# 3. (Optional) Download the iBot333 pretrained base checkpoint if specified
#    in config: vitl16_ibot333_highres512.yaml
```

In [ ]:
# %pip install -q torch torchvision Pillow timm

# Clone CheXFound repo if not already done
import subprocess, os

CHEXFOUND_DIR = "../../third_party_models/CheXFound/"
if not os.path.exists(CHEXFOUND_DIR):
    subprocess.run(["git", "clone", "https://github.com/RPIDIAL/CheXFound", CHEXFOUND_DIR],
                   check=True)
    print("Cloned CheXFound")
else:
    print("CheXFound already present")

print(os.listdir(CHEXFOUND_DIR))

In [ ]:
import os

# Control which GPU(s) are visible to this notebook.
# Set CUDA_VISIBLE_DEVICES in your shell before launching Jupyter to override,
# e.g.:  CUDA_VISIBLE_DEVICES=2 jupyter lab
# Or change the default value below.
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "6")
print(f"CUDA_VISIBLE_DEVICES = {os.environ['CUDA_VISIBLE_DEVICES']}")

## 1. Locate the checkpoint

Download from: https://drive.google.com/drive/folders/1GX2BWbujuVABtVpSZ4PTBykGULzrw806

In [ ]:
import os

# Update this path after downloading the checkpoint
CKPT_PATH = "../../third_party_models/CheXFound/weights/teacher_checkpoint.pth"

assert os.path.exists(CKPT_PATH), (
    f"Checkpoint not found at {CKPT_PATH}.\n"
    "Download from: https://drive.google.com/drive/folders/1GX2BWbujuVABtVpSZ4PTBykGULzrw806"
)
print("Checkpoint found:", os.path.getsize(CKPT_PATH) / 1e6, "MB")

## 2. Load the model

CheXFound uses a DINOv2/iBOT-style ViT-L backbone. The `teacher_checkpoint.pth` stores
the teacher EMA weights under the key `"teacher"` with a `"backbone."` prefix.

In [ ]:
import sys
sys.path.insert(0, CHEXFOUND_DIR)

import torch
from chexfound.models.vision_transformer import vit_large

# block_chunks=0 → flat blocks.N.* keys, matching the official key stripping below.
# Default block_chunks=1 wraps all blocks in one BlockChunk → nested blocks.0.N.* keys
# which don't match the 4-chunk FSDP checkpoint structure.
model = vit_large(patch_size=16, num_register_tokens=4, img_size=512,
                  ffn_layer="swiglufused", block_chunks=0)

# Load teacher EMA weights — official CheXFound key stripping
state_dict = torch.load(CKPT_PATH, map_location="cpu")["teacher"]
new_state_dict = {}
for k, v in state_dict.items():
    if k.startswith("backbone"):
        ls = k.split(".")
        if "blocks" in k:
            new_k = ".".join([ls[1], *ls[3:]])  # drop group index, keep block index
        else:
            new_k = ".".join(ls[1:])
    else:
        new_k = k
    new_state_dict[new_k] = v

msg = model.load_state_dict(new_state_dict, strict=False)
print("Missing keys:  ", msg.missing_keys)
print("Unexpected keys:", msg.unexpected_keys[:3], "..." if len(msg.unexpected_keys) > 3 else "")
model.eval()
print("Model ready.")

## 3. Inspect checkpoint keys (diagnostic — run if load fails)

In [ ]:
ckpt_keys = list(state.keys())
print("Total keys in checkpoint:", len(ckpt_keys))
print("First 10:", ckpt_keys[:10])
print("Last  10:", ckpt_keys[-10:])

model_keys = list(model.state_dict().keys())
print("\nFirst 10 model keys:", model_keys[:10])

## 4. Image transform

CheXFound input: 512×512, 3-channel, ImageNet normalization (standard for iBOT/DINOv2).

In [ ]:
from torchvision import transforms
from torchvision.transforms.functional import InterpolationMode
from PIL import Image

INPUT_SIZE = 512

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

def make_transform():
    return transforms.Compose([
        transforms.Resize(INPUT_SIZE, interpolation=InterpolationMode.BICUBIC),
        transforms.CenterCrop(INPUT_SIZE),
        transforms.ToTensor(),
        transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ])

transform = make_transform()

def preprocess(path: str) -> torch.Tensor:
    """Return [1, 3, 512, 512] batch tensor. Converts to RGB (grayscale → 3ch via PIL)."""
    img = Image.open(path).convert("RGB")
    return transform(img).unsqueeze(0)

dummy = torch.rand(1, 3, INPUT_SIZE, INPUT_SIZE)
print("dummy input:", dummy.shape)

## 5. Extract embeddings

In [ ]:
with torch.no_grad():
    cls_emb = model(dummy)   # DINOv2-style: forward() returns CLS token [B, 1024]

print("CLS embedding:", cls_emb.shape)   # [1, 1024]
print("L2 norm:", cls_emb.norm(dim=-1).item())

## 6. Get all patch tokens (for dense tasks / GLoRI head)

In [ ]:
# get_intermediate_layers strips CLS + register tokens from the output.
# Use return_class_token=True to get CLS separately as a (patches, cls) tuple.
with torch.no_grad():
    patches, cls = model.get_intermediate_layers(dummy, n=1, return_class_token=True)[0]

# patches: [B, N_patches, D]  — 512px / 16px = 32×32 = 1024 patches
# cls:     [B, D]
print("patch tokens:", patches.shape)   # [1, 1024, 1024]
print("CLS token:   ", cls.shape)       # [1, 1024]

## Segmentation mode — spatial feature map

Reshape the patch sequence from section 6 to a 2-D grid for dense prediction.
The 512×512 input with 16-px patches gives a **32×32** feature map.

In [ ]:
import math

# patches: [B, N_patches, D] — 512px / 16px-patches → 32×32 = 1024 patches, D=1024
B, N, D = patches.shape
H = W = int(math.sqrt(N))   # 32
assert H * W == N

spatial = patches.permute(0, 2, 1).reshape(B, D, H, W)
print(f"patch tokens:  {patches.shape}")   # [1, 1024, 1024]
print(f"spatial map:   {spatial.shape}")   # [1, 1024, 32, 32]

## Notes for RadHarmony integration

```
model_call : None (default model(x) returns CLS token directly)
pool       : None
embed_dim  : 1024
transform  : Resize(512,BICUBIC) → CenterCrop(512) → Grayscale → ToTensor
             → repeat(3ch) → Normalize(ImageNet)
```

**Checkpoint key prefix**: If `load_state_dict` reports many missing keys, inspect
the checkpoint structure in cell 3. Common issue: keys stored as `module.XXX` 
(DDP training) — strip with `k.replace('module.', '')`.

**GLoRI head**: CheXFound's key contribution is the GLoRI classification head
that fuses CLS + multi-scale local patch tokens. For linear-probe experiments,
just the CLS token (cell 5) is sufficient. For full GLoRI, see the repo's
`models/glori.py`.

**Segmentation mode** (`output_keys={"img", "mask"}`):
- Patch tokens: `model.get_intermediate_layers(x, n=1)[0][:, 1:]` → `[B, N, D]`
- Spatial map: reshape to `[B, 1024, 32, 32]` (512px / 16px-patches → 32×32)
- Use `patches.permute(0,2,1).reshape(B, D, H, W)` for the segmentation head

In [ ]:
CXR_PATH = "/path/to/cxr.jpg"

x = preprocess(CXR_PATH)
print("input:", x.shape, "range:", x.min().item(), x.max().item())

with torch.no_grad():
    emb = model(x)   # [1, 1024]

print("embedding:", emb.shape)
print("L2 norm:  ", emb.norm(dim=-1).item())

## Notes for RadHarmony integration

```
model_call : None (default model(x) returns CLS token directly)
pool       : None
embed_dim  : 1024
transform  : Resize(512,BICUBIC) → CenterCrop(512) → Grayscale → ToTensor
             → repeat(3ch) → Normalize(ImageNet)
```

**Checkpoint key prefix**: If `load_state_dict` reports many missing keys, inspect
the checkpoint structure in cell 3. Common issue: keys stored as `module.XXX` 
(DDP training) — strip with `k.replace('module.', '')`.

**GLoRI head**: CheXFound's key contribution is the GLoRI classification head
that fuses CLS + multi-scale local patch tokens. For linear-probe experiments,
just the CLS token (cell 5) is sufficient. For full GLoRI, see the repo's
`models/glori.py`.